# Business Analytics — Customer LTV & RFM Segmentation
## Notebook 1: Exploratory Data Analysis

**Business Question:** Who are our most valuable customers, and how is revenue distributed across the customer base?

**Dataset:** UCI Online Retail Dataset — 541,909 transactions from a UK-based e-commerce store (Dec 2010 – Dec 2011)

> **Setup:** Download `Online Retail.xlsx` from https://archive.ics.uci.edu/dataset/352/online+retail  
> Place the file in `data/raw/Online Retail.xlsx`

---
**Sections:**
1. Load & Explore Raw Data
2. Data Cleaning
3. Revenue Feature Engineering
4. Sales Overview
5. Revenue by Country
6. Sales Over Time
7. Top Products & Customers
8. Business Conclusions

## 1. Load & Explore Raw Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_excel('data/raw/Online Retail.xlsx', engine='openpyxl')

print(f'Shape: {df.shape}')
print(f'Date range: {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')
print(f'\nColumn types:')
print(df.dtypes)
df.head()

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Missing': missing, 'Pct (%)': missing_pct})[missing > 0])

print(f'\n=== Basic Stats ===')
print(f'Unique invoices   : {df["InvoiceNo"].nunique():,}')
print(f'Unique products   : {df["StockCode"].nunique():,}')
print(f'Unique customers  : {df["CustomerID"].nunique():,}')
print(f'Unique countries  : {df["Country"].nunique():,}')
print(f'\nQuantity range: {df["Quantity"].min()} → {df["Quantity"].max()}')
print(f'UnitPrice range: {df["UnitPrice"].min()} → {df["UnitPrice"].max()}')

## 2. Data Cleaning

Key issues to address:
- **Cancelled orders** — InvoiceNo starting with 'C' (negative quantities)
- **Missing CustomerID** — cannot compute RFM without customer identity
- **Invalid Quantity / UnitPrice** — zero or negative values outside returns

In [ ]:
print(f'Raw rows: {len(df):,}')

# Remove cancelled orders
cancelled_mask = df['InvoiceNo'].astype(str).str.startswith('C')
print(f'Cancelled orders  : {cancelled_mask.sum():,} rows removed')
df = df[~cancelled_mask]

# Remove rows without CustomerID (can't attribute to a customer)
missing_customer = df['CustomerID'].isnull()
print(f'Missing CustomerID: {missing_customer.sum():,} rows removed')
df = df[~missing_customer]

# Remove invalid quantities and prices
invalid_qty = df['Quantity'] <= 0
invalid_price = df['UnitPrice'] <= 0
print(f'Invalid Quantity  : {invalid_qty.sum():,} rows removed')
print(f'Invalid UnitPrice : {invalid_price.sum():,} rows removed')
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Clean CustomerID type
df['CustomerID'] = df['CustomerID'].astype(int).astype(str)

print(f'\nClean rows: {len(df):,}')
print(f'Customers retained: {df["CustomerID"].nunique():,}')

## 3. Revenue Feature Engineering

In [ ]:
# Core revenue metric
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Time features
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Year']  = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()

total_revenue = df['Revenue'].sum()
print(f'Total Revenue   : £{total_revenue:,.0f}')
print(f'Avg Order Value : £{df.groupby("InvoiceNo")["Revenue"].sum().mean():,.2f}')
print(f'Avg Revenue/Cust: £{df.groupby("CustomerID")["Revenue"].sum().mean():,.2f}')

# Save clean data for next notebooks
import os
os.makedirs('data', exist_ok=True)
df.to_csv('data/online_retail_clean.csv', index=False)
print('\nSaved: data/online_retail_clean.csv')

## 4. Sales Overview

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Revenue distribution (log scale — heavy tail)
cust_revenue = df.groupby('CustomerID')['Revenue'].sum().sort_values(ascending=False)
axes[0, 0].hist(cust_revenue, bins=80, color='#4C72B0', alpha=0.8, edgecolor='white')
axes[0, 0].set_title('Revenue Distribution per Customer', fontweight='bold')
axes[0, 0].set_xlabel('Total Revenue (£)')
axes[0, 0].set_ylabel('Number of Customers')
axes[0, 0].set_yscale('log')

# Pareto — top X% of customers → Y% of revenue
cum_revenue = cust_revenue.cumsum() / cust_revenue.sum() * 100
cum_customers = np.arange(1, len(cust_revenue) + 1) / len(cust_revenue) * 100
axes[0, 1].plot(cum_customers, cum_revenue, color='#DD8452', linewidth=2)
axes[0, 1].axhline(80, color='gray', linestyle='--', linewidth=1)
axes[0, 1].axvline(20, color='gray', linestyle='--', linewidth=1)
axes[0, 1].set_title('Pareto Curve — Customer Revenue Concentration', fontweight='bold')
axes[0, 1].set_xlabel('% of Customers (sorted by revenue)')
axes[0, 1].set_ylabel('% of Cumulative Revenue')
axes[0, 1].fill_between(cum_customers, cum_revenue, alpha=0.1, color='#DD8452')

# Orders by day of week
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_revenue = df.groupby('DayOfWeek')['Revenue'].sum().reindex(dow_order)
axes[1, 0].bar(dow_revenue.index, dow_revenue.values / 1000, color='#55A868', alpha=0.85)
axes[1, 0].set_title('Revenue by Day of Week', fontweight='bold')
axes[1, 0].set_ylabel('Revenue (£ thousands)')
axes[1, 0].tick_params(axis='x', rotation=30)

# Orders by hour
df['Hour'] = df['InvoiceDate'].dt.hour
hour_revenue = df.groupby('Hour')['Revenue'].sum()
axes[1, 1].bar(hour_revenue.index, hour_revenue.values / 1000, color='#8172B2', alpha=0.85)
axes[1, 1].set_title('Revenue by Hour of Day', fontweight='bold')
axes[1, 1].set_xlabel('Hour')
axes[1, 1].set_ylabel('Revenue (£ thousands)')

plt.suptitle('Sales Overview — UCI Online Retail', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Pareto insight
top20_pct = cum_revenue.iloc[int(len(cum_revenue) * 0.20)]
print(f'Top 20% of customers generate {top20_pct:.1f}% of total revenue')

## 5. Revenue by Country

In [ ]:
country_revenue = (
    df.groupby('Country')
    .agg(Revenue=('Revenue', 'sum'), Customers=('CustomerID', 'nunique'), Orders=('InvoiceNo', 'nunique'))
    .sort_values('Revenue', ascending=False)
    .head(10)
)
country_revenue['Revenue_pct'] = (country_revenue['Revenue'] / country_revenue['Revenue'].sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['#DD8452' if i == 0 else '#4C72B0' for i in range(len(country_revenue))]
axes[0].barh(country_revenue.index[::-1], country_revenue['Revenue'][::-1] / 1000,
             color=colors[::-1], alpha=0.85)
axes[0].set_title('Top 10 Countries by Revenue', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Revenue (£ thousands)')

# Revenue ex-UK to show international spread
intl = country_revenue[country_revenue.index != 'United Kingdom']
axes[1].pie(intl['Revenue'], labels=intl.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(intl)))
axes[1].set_title('International Revenue Distribution (excl. UK)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(country_revenue[['Revenue', 'Customers', 'Orders', 'Revenue_pct']].to_string())

## 6. Sales Over Time

In [ ]:
monthly = (
    df.groupby('YearMonth')
    .agg(
        Revenue=('Revenue', 'sum'),
        Orders=('InvoiceNo', 'nunique'),
        Customers=('CustomerID', 'nunique')
    )
    .reset_index()
)
monthly['YearMonth_str'] = monthly['YearMonth'].astype(str)

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

axes[0].plot(monthly['YearMonth_str'], monthly['Revenue'] / 1000,
             color='#4C72B0', linewidth=2, marker='o', markersize=4)
axes[0].fill_between(monthly['YearMonth_str'], monthly['Revenue'] / 1000,
                     alpha=0.15, color='#4C72B0')
axes[0].set_title('Monthly Revenue (£ thousands)', fontweight='bold')
axes[0].set_ylabel('Revenue (£k)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(monthly['YearMonth_str'], monthly['Orders'],
            color='#55A868', alpha=0.85)
axes[1].set_title('Monthly Orders', fontweight='bold')
axes[1].set_ylabel('Number of Orders')
axes[1].tick_params(axis='x', rotation=45)

axes[2].plot(monthly['YearMonth_str'], monthly['Customers'],
             color='#DD8452', linewidth=2, marker='s', markersize=4)
axes[2].set_title('Monthly Active Customers', fontweight='bold')
axes[2].set_ylabel('Unique Customers')
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Business Trends — Dec 2010 to Dec 2011', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Top Products & Customers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 products by revenue
top_products = (
    df.groupby('Description')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(15)
)
axes[0].barh(top_products.index[::-1], top_products.values[::-1] / 1000,
             color='#4C72B0', alpha=0.85)
axes[0].set_title('Top 15 Products by Revenue', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Revenue (£ thousands)')

# Top 15 customers by revenue
top_customers = (
    df.groupby('CustomerID')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(15)
)
colors_cust = ['#DD8452' if i < 3 else '#4C72B0' for i in range(len(top_customers))]
axes[1].bar(range(len(top_customers)), top_customers.values / 1000,
            color=colors_cust, alpha=0.85)
axes[1].set_xticks(range(len(top_customers)))
axes[1].set_xticklabels([f'C{c}' for c in top_customers.index], rotation=45, ha='right')
axes[1].set_title('Top 15 Customers by Revenue', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Revenue (£ thousands)')

plt.tight_layout()
plt.show()

print(f'Top 3 customers account for £{top_customers.head(3).sum():,.0f} combined')
print(f'Top 3 as % of total revenue: {top_customers.head(3).sum() / df["Revenue"].sum() * 100:.1f}%')

## 8. Business Conclusions

### Key Findings

**Revenue Concentration (Pareto Effect)**
- The top 20% of customers drive ~80% of total revenue — a textbook Pareto distribution
- Revenue distribution is heavily right-skewed: most customers spend little, a few spend a lot
- This means **retaining top customers is disproportionately more valuable** than acquiring new ones

**Geographic Distribution**
- United Kingdom dominates revenue (>80%)
- Netherlands, EIRE, and Germany are the top international markets
- International customers represent a growth opportunity with existing infrastructure

**Temporal Patterns**
- Revenue peaks in **November** (holiday season buildup) then drops sharply in December
- **Thursday and Tuesday** are the highest-revenue days — align promotions accordingly
- Peak hours are **10am–14:00** — optimal window for email campaigns

**Product Concentration**
- A small number of products drive a large share of revenue
- Top products are decorative/gift items — consistent with the holiday seasonality

### Business Implication
Revenue is highly concentrated — in customers, products, and time. The strategic priority is:
1. **Identify and protect** high-value customers before they churn
2. **Reactivate** dormant customers who have recently stopped buying
3. **Develop** mid-tier customers into high-value ones

→ **Next step:** RFM Segmentation to classify every customer by value and risk level (`02_rfm_segmentation.ipynb`)